In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# ==========================================================
# 1. Load YOLO model
# ==========================================================
model = YOLO("runs/train/toy_animals_full/weights/best.pt")

# ==========================================================
# 2. CAMERA ACCESS (Dofbot camera)
# Change this depending on your robot camera
# ==========================================================
CAMERA_INDEX = 0  # <-- Change if needed (0,1,2...)
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise Exception("❌ ERROR: Cannot open Dofbot camera.")

print("✔ Dofbot camera detected")

# ==========================================================
# 3. CAMERA INTRINSIC CALIBRATION (Checkerboard method)
# ==========================================================
def calibrate_intrinsics():
    """
    Uses checkerboard calibration to compute camera matrix & distortion.
    Run this ONCE and save the parameters.
    """
    CHECKERBOARD = (6, 9)       # number of internal corners
    obj_points = []
    img_points = []

    objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

    print("📸 Starting camera intrinsic calibration... Press 'c' to capture frames.")

    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        ret_cb, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

        # Show checkerboard detection
        disp = frame.copy()
        if ret_cb:
            cv2.drawChessboardCorners(disp, CHECKERBOARD, corners, ret_cb)

        cv2.imshow("Calibration", disp)
        key = cv2.waitKey(1)

        # Capture frame when pressing "c"
        if key == ord("c") and ret_cb:
            print("✔ Captured checkerboard frame")
            obj_points.append(objp)
            img_points.append(corners)

        # Quit calibration
        if key == ord("q"):
            break

    cv2.destroyWindow("Calibration")

    # Compute intrinsic parameters
    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        obj_points, img_points, gray.shape[::-1], None, None
    )

    print("\n=== CAMERA INTRINSICS ===")
    print("Camera Matrix (K):\n", K)
    print("\nDistortion Coefficients:\n", dist)

    return K, dist


# ==========================================================
# 4. CAMERA → ROBOT TRANSFORM (4x4 matrix)
# This MUST be measured manually ONCE
# ==========================================================
def get_camera_to_robot_transform():
    """
    YOU MUST MEASURE THIS.
    Place an ArUco marker at a known robot coordinate (Xr, Yr, Zr).
    Detect marker → get camera coords → solve transform.
    Example below uses placeholder.
    """

    T = np.array([
        [1, 0, 0, 120],   # X translation (mm)
        [0, 1, 0, -30],   # Y translation
        [0, 0, 1, 180],   # Z translation
        [0, 0, 0, 1]
    ], dtype=float)

    print("\n✔ Loaded Camera-to-Robot Transform:")
    print(T)
    return T


# ==========================================================
# 5. 2D PIXEL → 3D ROBOT COORDINATES
# Requires intrinsics + transform
# ==========================================================
def pixel_to_robot(x_pix, y_pix, depth, K, T_cam_to_robot):
    fx = K[0, 0]
    fy = K[1, 1]
    cx = K[0, 2]
    cy = K[1, 2]

    # WORK BACKWARD TO GET CAMERA COORDINATES
    Xc = (x_pix - cx) * depth / fx
    Yc = (y_pix - cy) * depth / fy
    Zc = depth

    # Homogeneous → 4x1 vector
    P_cam = np.array([Xc, Yc, Zc, 1])

    # Transform to robot frame
    P_robot = T_cam_to_robot @ P_cam

    return P_robot[:3]  # return (X, Y, Z) in mm


# ==========================================================
# 6. COLOR PROCESSING
# ==========================================================
def get_dominant_color(image):
    pixels = image.reshape((-1, 3))
    pixels = np.float32(pixels)

    _, labels, palette = cv2.kmeans(
        pixels, 2, None,
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 0.2),
        10,
        cv2.KMEANS_RANDOM_CENTERS
    )

    _, counts = np.unique(labels, return_counts=True)
    return palette[np.argmax(counts)].astype(int)


def color_name(rgb):
    r, g, b = rgb
    hsv = cv2.cvtColor(np.uint8([[rgb]]), cv2.COLOR_BGR2HSV)[0][0]
    h, s, v = hsv

    if s < 40 and v > 160:
        return "white/black (zebra)"
    if s < 50 and 50 < v < 200:
        return "grey"
    if 20 < h < 35 and s > 80:
        return "yellow"
    if (5 < h < 20 and s > 100) or (h <= 5 and s > 100):
        return "orange/red"
    return "unknown"


# ==========================================================
# 7. MAIN LOOP (YOLO + CALIBRATION + ROBOT COORDINATES)
# ==========================================================

# Load calibration (you must run calibration once)
K, dist = calibrate_intrinsics()
T_cam_to_robot = get_camera_to_robot_transform()

print("\n🎉 Calibration complete. Starting YOLO detection...\n")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Remove lens distortion
    frame = cv2.undistort(frame, K, dist)

    # Run YOLO on the frame
    results = model(frame)
    annotated = results[0].plot()

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        label = model.names[cls]

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        # For now: assume object is on table (constant height)
        ESTIMATED_DEPTH_MM = 200  # <-- CHANGE if object is on table or use depth camera

        # Convert to robot coordinates
        Xr, Yr, Zr = pixel_to_robot(cx, cy, ESTIMATED_DEPTH_MM, K, T_cam_to_robot)

        # Get dominant color
        crop = frame[y1:y2, x1:x2]
        col = "?"
        if crop.size > 0:
            col_rgb = get_dominant_color(crop)
            col = color_name(col_rgb)

        text = f"{label} ({col})  Robot XYZ = {Xr:.1f}, {Yr:.1f}, {Zr:.1f}"
        cv2.putText(annotated, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Show
    cv2.imshow("YOLO + Calibration + Dofbot", annotated)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
